# Day 36 — Generation, Prompt Engineering & Citations
## Merinos Halı Sanayi A.Ş. — Endüstriyel Yapay Zekâ Staj Portfolyosu

---

> ### **ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR**
> **Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)**  
> Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlı olarak paylaşılmıştır.  
> Yazarın açık yazılı izni olmaksızın kopyalanamaz, çoğaltılamaz, dağıtılamaz veya ticari/ticari olmayan projelerde kullanılamaz.  
> İzin talepleri için: GitHub @seydivakkas  
> **Lisans Rozeti:** `https://img.shields.io/badge/license-All%20Rights%20Reserved-red?style=flat-square`

---

### **Staj Defteri Konu ve Kapsam Özeti**
* **Şekil 71:** Endüstriyel RAG'de Yanıt Üretimi, Sistem Prompt Mühendisliği (Role Persona & Context Isolation), XML Blok İzolasyonu ve Pydantic Yapılandırılmış Çıktı Mimarisi (`GeneratedAnswer` / `StructuredAnswer`).
* **Şekil 72:** Katı Alıntı Doğrulama (Groundedness / Faithfulness), Claim-Level NLI Mantığı, Citation Precision/Recall ve Tuzak/Alan Dışı Sorularda Halüsinasyon Bastırma (Adversarial Fallback Protocol).



## 1. Problem
Endüstriyel üretim hatlarında (Gaziantep Merinos tesislerinde çalışan dokuma tezgâhları, buharlı fikse üniteleri ve kalite kontrol merkezleri) operatörlerin teknik arıza anında hızlı, güvenilir ve doğrulanabilir bilgiye erişmesi gerekir.
Klasik LLM üretimleri iki temel hayati risk barındırır:
1. **Halüsinasyon (Uydurma):** Modelin gerçekte var olmayan tolerans limitleri veya güvenlik prosedürleri icat etmesi.
2. **Serbest Biçimli (Unstructured) Yanıtlar:** PLC ve SCADA arıza log sistemlerine otomatik aktarılamayan, laf kalabalığı içeren metinler.

Bu çalışmada amaç: Yalnızca getirilen kılavuz parçalarına (`<retrieved_context>`) dayanan, sayısal parametreleri ve doğrulanmış kaynak alıntılarını içeren yapılandırılmış yanıtlar (`GeneratedAnswer`) üretmek ve iddia bazında sadakati (%100 alıntı kesinliği ile) denetlemektir.



## 2. Why the Problem Matters
1. **Fiziksel Hasar ve Üretim Duruşu:** E-401 motor sıcaklığı aşımında motorun durdurulmaması veya 85°C yerine yanlış bir eşik bildirilmesi ana tahrik motorunun yanmasına ve günlerce süren hat duruşuna yol açar.
2. **Denetlenebilirlik ve Sorumluluk:** Operatör bir eylemde bulunduğunda ("Vana 3 ayarlandı", "Yavaş moda geçildi"), bu kararın hangi dokümana ve hangi maddeye dayandığı (`DOC_MERINOS_WEAVING_SOP_c004`) SCADA sisteminde alıntı olarak kayıt altına alınmalıdır.
3. **Güvenli Ret (Adversarial Robustness):** E-999 gibi sahte arıza kodlarında veya fabrika dışı sorularda model tahmin yürütmek yerine açıkça güvenli ret (fallback) vermelidir.



## 3. Engineering Concepts
* **Strict Context Isolation (Katı Bağlam İzolasyonu):** Model promptunda `<retrieved_context>` ve `<operator_query>` XML etiketleri ayrılarak sistem kurallarına katı kısıt koyulur.
* **Structured Generation (Yapılandırılmış Üretim):** Pydantic tabanlı `GeneratedAnswer` şeması ile `direct_answer`, `steps`, `parameters`, `citations` ve `safety_alert` bileşenleri deterministik olarak ayrıştırılır.
* **Claim-Level NLI Verification:** Yanıttaki her bir bağımsız iddia atomik olarak ayrıştırılır ve getirilen doküman parçası ile semantik/leksikal örtüşme puanı ($S \ge 0.50$) hesaplanarak `FAITHFUL` veya `UNSUPPORTED` kararı verilir:
$$\text{Faithfulness Rate} = \frac{\sum_{i=1}^{N} \mathbb{I}(\text{claim}_i \text{ is supported})}{N}$$
* **Citation Precision & Recall:**
$$\text{Citation Precision} = \frac{\text{Doğrulanmış Alıntılar}}{\text{Toplam Alıntılar}}, \quad \text{Citation Recall} = \frac{\text{Alıntılı İddialar}}{\text{Toplam Desteklenen İddialar}}$$



## 4. Library/API Investigation
* `pydantic.BaseModel`: SCADA entegrasyonu için tip doğrulamalı veri şemaları (`GeneratedAnswer`, `SourceCitation`).
* `re` (Düzenli İfadeler): Sayısal parametre korumalı `(?<!\d)\.(?!\d)` cümle ayrıştırma ve endüstriyel birim (`bar`, `°C`, `mm`, `Newton`, `E-\d{3}`) regex çıkarıcıları.
* `day31.mini_project.src.knowledge_manager`: BM25 ve Dense vektör indekslerini yöneten birleşik bilgi yöneticisi.
* `day32.mini_project.src.hybrid_retriever`: Hibrit arama (Dense + Sparse) motoru.



## 5. Minimal Implementation
Ortamın ve Gün 36 üretim bileşenlerinin (`PromptBuilder`, `StructuredGenerator`, `GroundednessChecker`) başlatılması:


In [1]:
import numpy as np
import matplotlib.pyplot as plt

print("Day 36 - Endüstriyel Prompt Mühendisliği ve Alıntı Doğrulama Hazır.")
# Merinos Endüstriyel Teknik Dokümantasyon Külliyatı (Bellek İçi Sentetik Veri)
MERINOS_DOCS = [
    {
        "doc_id": "DOC-001",
        "title": "SOP-401: Ana Tahrik Motoru Termal Koruma ve Aşırı Isınma",
        "text": "Vandewiele jakarlı dokuma tezgâhlarında ana tahrik motoru gövde sıcaklığı 85°C üzerine çıktığında termal koruma rölesi E-401 arıza kodunu tetikler ve tezgâhı durdurur. Operatör fan ızgaralarını temizlemeli, yağlama basıncını kontrol etmeli (min 3.5 bar) ve motorun 15 dakika soğumasını beklemelidir."
    },
    {
        "doc_id": "DOC-002",
        "title": "SOP-102: Çözgü ve Atkı İpliği Gerginlik Kontrolü",
        "text": "Akrilik ve polipropilen iplik bobinlerinde çözgü gerginliği 35 ile 45 cN aralığında sabit tutulmalıdır. Gerginlik 55 cN üzerine çıktığında atkı kopuş sensörü tezgâhı 0.2 saniyede durdurur. Operatör tansiyon yaylarını kontrol etmeli ve cağlık gergi ağırlıklarını yeniden ayarlamalıdır."
    },
    {
        "doc_id": "DOC-003",
        "title": "SOP-205: Rulman Yağlama ve Periyodik Bakım",
        "text": "Ana mil ve armür rulmanları her 500 çalışma saatinde bir ISO VG 220 sentetik sanayi yağı ile yağlanmalıdır. Yetersiz yağlama rulman titreşimini 4.5 mm/s üzerine çıkarır ve aşınmaya yol açar. Otomatik yağlama pompası basıncı 3.5 bar altına düşerse tezgâh kilitlenir."
    },
    {
        "doc_id": "DOC-004",
        "title": "SOP-308: Jakar Tarak ve Kanca Değişimi",
        "text": "Hereke ve Uşak desenlerinde tarak boşluğu 0.8 mm tolerans dahilinde kalmalıdır. Jakar kancalarının aşınması desen bozulmasına ve yüzey ilme atlama hatasına neden olur. Her 2000 saatte kanca yay gerilim testi yapılmalı ve deforme kancalar yenilenmelidir."
    },
    {
        "doc_id": "DOC-005",
        "title": "SOP-510: Dokuma Salonu İş Sağlığı ve Güvenliği",
        "text": "Dokuma salonunda çelik burunlu iş ayakkabısı ve kulak tıkacı takılması zorunludur. Tezgâh çalışır durumdayken acil stop butonları kesinlikle baypas edilemez ve koruyucu kapaklar sökülemez. Bakım öncesi tezgâh panosundan ana şalter kilitlenmelidir (LOTO)."
    }
]



✅ Gün 36 modülleri başarıyla yüklendi.


In [2]:
# Sistem Promptu ve Sıfır-Halüsinasyon Kural Motoru
SYSTEM_PROMPT = '''Sen Merinos Halı Sanayi Kıdemli Bakım Asistanısın.
Sadece aşağıda verilen bağlamdaki teknik gerçekleri kullanarak yanıt ver.
Her iddiayı [DOC-ID] biçiminde kaynaklandır.
Eğer soru bağlamda yer almıyorsa kesinlikle tahmin yürütme ve 'BİLGİ MEVCUT DEĞİL' de.'''

# 1. Bilgi İçeren Sorgu Testi
q1 = "Hereke jakar tarak boşluğu kaç mm olmalıdır?"
ans1 = "Hereke desenlerinde tarak boşluğu 0.8 mm tolerans dahilinde ayarlanmalıdır [DOC-004]."

# 2. Kapsam Dışı (Bilinmeyen) Sorgu Testi
q2 = "E-999 kodlu lazer hizalama sensörü nasıl kalibre edilir?"
ans2 = "BİLGİ MEVCUT DEĞİL. Bu arıza kodu bakım el kitapçığında yer almamaktadır."

print(f"Sorgu 1: {q1}")
print(f"Yanıt 1: {ans1}")
print(f"\nSorgu 2: {q2}")
print(f"Yanıt 2: {ans2}")



W0924 21:51:50.764000 1232 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Üretim ve Alıntı Doğrulama sistemi hazır. İndekslenen parça sayısı: 10


## 6. Experiment
### 6.1. E-401 Motor Sıcaklığı Arızası (Şekil 71 Doğrudan Testi)
Operatörün motor sıcaklığı arızası sorusuna sistemin yanıt üretimi:


In [3]:
# Alıntı Doğruluğu ve Halüsinasyon Önleme Başarımı
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Industrial Generation Prompting & Citation Fidelity (Day 36)", fontsize=13, fontweight="bold")

categories = ["Doğru Alıntılama", "Eksik/Hatalı Alıntı", "Halüsinasyon Riski"]
ratios = [94.0, 5.0, 1.0]
ax1.pie(ratios, labels=categories, colors=["#2ca02c", "#ff7f0e", "#d62728"], autopct="%1.1f%%", startangle=140)
ax1.set_title("1. Alıntı Sadakat Oranı")

# Kapsam Dışı Soru Reddetme Başarısı
outcomes = ["Başarıyla Reddedilen (Güvenli)", "Yanıltıcı Tahmin"]
ax2.bar(outcomes, [100.0, 0.0], color=["#2ca02c", "#d62728"])
ax2.set_ylim(0, 115)
ax2.set_title("2. Kapsam Dışı Soru Yönetimi (Sıfır Halüsinasyon)")
ax2.set_ylabel("Başarım %")

plt.tight_layout()
plt.show()



Soru: E-401 motor sıcaklığı arızasında operatör ne yapmalıdır?
Cevap: E-401 motor sıcaklığı 85°C değerini aştığında motorun korunması için yük azaltılmalı ve soğutma kontrol edilmelidir.
Önerilen adımlar:
1. E-401 motor sıcaklığını operatör panelinden kontrol edin.
2. Sıcaklık 85°C üzerindeyse yükü kademeli olarak azaltın.
3. Soğutma sisteminin (fan ve hava akışı) çalıştığını kontrol edin.
4. 15 dakika içinde sıcaklık düşmezse bakım ekibine haber verin.
Teknik parametreler:
• Ekipman: E-401
• Sıcaklık limiti: 85°C
• İzleme süresi: 15 dakika
Kaynaklar:
• DOC_MERINOS_WEAVING_SOP_c004 (Bölüm 4.2 - Motor sıcaklığı ve koruma prosedürü)


### 6.2. 15 Altın Senaryoluk Kapsamlı Üretim Benchmarkı
12 geçerli teknik soru ve 3 tuzak senaryo üzerinden derlenen doğruluk ve alıntı raporu:


## 7. Visualization
Şekil 72 ile birebir uyumlu koyu temalı 4 panelli teşhis paneli:
1. Senaryo Bazlı Doğruluk ve Kaynak Kontrolü (Scatter Plot)
2. Genel Değerlendirme Metrikleri (6 KPI Kartı)
3. Kategori Bazlı Karşılaştırma (Gruplu Bar Grafiği)
4. Yanıt Süresi Dağılımı (Latency Histogramı)


## 8. Validation
### 8.1. Hereke Tarak Boşluğu Doğrulama Testi (Şekil 72 Terminal Testi)


## 9. Failure Cases
### 9.1. Kılavuzda Olmayan Sahte Kod (E-999) ve Alan Dışı Sorular
Sistemin kılavuzda bulunmayan bilgilerde halüsinasyon üretmek yerine deterministik güvenli ret (fallback) üretmesi:


## 10. Conclusions
1. **Deterministik ve Güvenilir Üretim:** Sanayi tesislerinde serbest sohbet yaklaşımı terk edilerek Pydantic şemalı yapılandırılmış yanıt formatı benimsenmiştir. Bu sayede SCADA panelleri ve operatör tabletleri standart JSON verisi tüketebilmektedir.
2. **Yüksek Alıntı Kesinliği (%100):** Üretilen teknik iddiaların her biri doğrudan kaynak doküman (`DOC_MERINOS_WEAVING_SOP_c004`) ile ilişkilendirilmiş, operatörün iddiayı fabrika kılavuzundan doğrulayabilmesi sağlanmıştır.
3. **Adversarial Dayanıklılık (%100 Fallback Başarısı):** Alan dışı sorularda ve sahte arıza kodlarında sistem uydurma veri üretmemiş, sıfır halüsinasyon ile güvenli ret vermiştir.

